# PEST-R checks: literature, references, licences (CPU only, about 5-10 min)
Settings: Accelerator = None, **Internet = ON**. No token is needed. Nothing is trained. Send me the output of cells 3, 4 and 5 (or the three .txt files).

In [ ]:
!pip -q install pypdf 2>&1 | tail -1

In [ ]:
%%writefile /kaggle/working/checks_lib.py
# Helper functions used by pest_checks.ipynb (Kaggle, Internet ON, CPU). Read-only: nothing is trained, nothing is overwritten.
import re, io, json, time, difflib, urllib.request, urllib.parse

UA = {"User-Agent": "pest-r-checks/1.0"}                    # generic client name, no personal contact data
def http(url, timeout=60):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()

# ---------------------------------------------------------------- 1) literature: grep the ORIGINAL PDFs, appendices included
PAPERS = {
 "ROHE / oDPO (NAACL 2025)":      "https://aclanthology.org/2025.naacl-long.349.pdf",
 "POPEv2 / Obliviate (2508.04567)": "https://arxiv.org/pdf/2508.04567",
 "HII-DPO (2602.10425)":          "https://arxiv.org/pdf/2602.10425",
 "HalluSegBench (2506.21546)":    "https://arxiv.org/pdf/2506.21546",
}
KEYS = ["placebo", "control edit", "irrelevant region", "unrelated region", "random region", "random location", "different location", "other region",
        "edit artifact", "editing artifact", "artifact", "shortcut", "spurious", "edited image", "edit cue", "inpainting trace", "same size", "same shape",
        "sanity check", "negative control", "confound", "learn to detect", "tell whether"]
def pdf_text(url):
    from pypdf import PdfReader
    r = PdfReader(io.BytesIO(http(url, 120))); return [(i + 1, (p.extract_text() or "")) for i, p in enumerate(r.pages)]
def grep_papers(papers=PAPERS, keys=KEYS, ctx=170):
    out = []
    for name, url in papers.items():
        out.append("=" * 110); out.append(name + "\n" + url)
        try: pages = pdf_text(url)
        except Exception as e: out.append(f"   COULD NOT FETCH/READ: {e}\n   -> open the PDF yourself and search for these words: {', '.join(keys[:8])}"); continue
        full = "\n".join(t for _, t in pages)
        app = [n for n, t in pages if re.search(r"^\s*(appendix|a\s+appendix|supplementary)", t, re.I | re.M)]
        out.append(f"   pages: {len(pages)}   characters: {len(full)}   pages that start an appendix/supplement: {app or 'none detected'}")
        if len(full) < 5000: out.append("   WARNING: very little text extracted (scanned or protected PDF?) - search it by hand.")
        for k in keys:
            hits = [(n, m.start()) for n, t in pages for m in re.finditer(re.escape(k), t, re.I)]
            if not hits: continue
            out.append(f"   [{k}]  {len(hits)} hit(s)")
            for n, _ in hits[:4]:
                t = dict(pages)[n]; m = re.search(re.escape(k), t, re.I)
                out.append(f"      p.{n}: ..." + re.sub(r"\s+", " ", t[max(0, m.start() - ctx): m.end() + ctx]) + "...")
    return "\n".join(out)

# ---------------------------------------------------------------- 2) references: compare every BibTeX entry with Crossref / arXiv
def _field(body, name):
    m = re.search(r"\b" + name + r"\s*=\s*\{", body, re.I)
    if not m: return ""
    i, depth = m.end(), 1
    while i < len(body) and depth:
        depth += (body[i] == "{") - (body[i] == "}"); i += 1
    return body[m.end(): i - 1]
def parse_bib(txt):
    ents = []
    for chunk in re.split(r"(?m)^(?=@\w+\{)", txt):
        m = re.match(r"@(\w+)\{([^,]+),", chunk)
        if not m: continue
        body = chunk[m.end():]; g = lambda f: re.sub(r"[{}]", "", _field(body, f)).strip()
        ents.append(dict(key=m.group(2).strip(), type=m.group(1), title=g("title"), year=g("year"), venue=g("journal") or g("booktitle"), author=g("author"), body=body))
    return ents
def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower())
def check_ref(e):
    t = e["title"]; res = {"key": e["key"], "bib_title": t, "bib_year": e["year"], "bib_venue": e["venue"]}
    am = re.search(r"arXiv:(\d{4}\.\d{4,5})", e["body"])
    try:
        if am and "arxiv" in e["venue"].lower():
            x = http("https://export.arxiv.org/api/query?id_list=" + am.group(1)).decode("utf8", "ignore")
            mt = re.search(r"<title>(.*?)</title>.*?<published>(\d{4})", x.split("<entry>", 1)[1], re.S)
            res.update(found_title=re.sub(r"\s+", " ", mt.group(1)).strip(), found_year=mt.group(2), found_venue="arXiv " + am.group(1), src="arXiv")
        else:
            j = json.loads(http("https://api.crossref.org/works?rows=2&select=title,issued,container-title,author&query.bibliographic=" + urllib.parse.quote(t + " " + e["author"].split(" and ")[0])))
            it = j["message"]["items"][0]
            res.update(found_title=(it.get("title") or [""])[0], found_year=str((it.get("issued", {}).get("date-parts") or [[""]])[0][0]),
                       found_venue=(it.get("container-title") or [""])[0], src="Crossref")
    except Exception as ex:
        res.update(found_title="", found_year="", found_venue="", src=f"lookup failed: {ex}")
    sim = difflib.SequenceMatcher(None, norm(t), norm(res["found_title"])).ratio() if res["found_title"] else 0
    ydiff = abs(int(res["found_year"]) - int(e["year"])) if res["found_year"].isdigit() and e["year"].isdigit() else None
    res["similarity"] = round(sim, 2)
    res["flag"] = "OK" if sim >= 0.9 and (ydiff is None or ydiff <= 1) else ("CHECK BY HAND (title or year differs)" if res["found_title"] else "NOT FOUND - check by hand")
    return res
def check_all(bibtxt, pause=0.4):
    out = []
    for e in parse_bib(bibtxt): out.append(check_ref(e)); time.sleep(pause)
    return out

# ---------------------------------------------------------------- 3) licences of the models actually used (from the Hugging Face model cards)
MODELS = ["llava-hf/llava-1.5-7b-hf", "Qwen/Qwen2.5-VL-3B-Instruct", "Qwen/Qwen2.5-VL-7B-Instruct", "HuggingFaceTB/SmolVLM-Instruct",
          "google/owlvit-base-patch32", "stable-diffusion-v1-5/stable-diffusion-inpainting"]
def licences(token=None):
    from huggingface_hub import model_info
    out = []
    for m in MODELS:
        try:
            i = model_info(m, token=token); cd = i.card_data.to_dict() if i.card_data else {}
            out.append((m, str(cd.get("license")), str(cd.get("license_name") or ""), str(cd.get("license_link") or ""), "gated" if i.gated else "open"))
        except Exception as e: out.append((m, "LOOKUP FAILED", str(e)[:80], "", ""))
    return out


## 3. Literature: search the original PDFs (appendices included) for placebo / control-edit language

In [ ]:
import sys; sys.path.insert(0, '/kaggle/working')
import checks_lib as C
rep = C.grep_papers()
open('/kaggle/working/literature_grep.txt', 'w').write(rep); print(rep)

## 4. References: compare every BibTeX entry with Crossref / arXiv

In [ ]:
%%writefile /kaggle/working/refs.bib
@article{geirhos2020,
  author={Geirhos, Robert and Jacobsen, J{\"o}rn-Henrik and Michaelis, Claudio and Zemel, Richard and Brendel, Wieland and Bethge, Matthias and Wichmann, Felix A.},
  title={Shortcut learning in deep neural networks}, journal={Nature Machine Intelligence}, volume={2}, number={11}, pages={665--673}, year={2020}}
@inproceedings{li2023pope,
  author={Li, Yifan and Du, Yifan and Zhou, Kun and Wang, Jinpeng and Zhao, Wayne Xin and Wen, Ji-Rong},
  title={Evaluating object hallucination in large vision-language models}, booktitle={Proc. EMNLP}, pages={292--305}, year={2023}}
@article{fu2023mme,
  author={Fu, Chaoyou and Chen, Peixian and Shen, Yunhang and Qin, Yulei and Zhang, Mengdan and Lin, Xu and others},
  title={{MME}: A comprehensive evaluation benchmark for multimodal large language models}, journal={arXiv preprint arXiv:2306.13394}, year={2023}}
@inproceedings{rohrbach2018,
  author={Rohrbach, Anna and Hendricks, Lisa Anne and Burns, Kaylee and Darrell, Trevor and Saenko, Kate},
  title={Object hallucination in image captioning}, booktitle={Proc. EMNLP}, pages={4035--4045}, year={2018}}
@inproceedings{liu2024llava15,
  author={Liu, Haotian and Li, Chunyuan and Li, Yuheng and Lee, Yong Jae},
  title={Improved baselines with visual instruction tuning}, booktitle={Proc. IEEE/CVF CVPR}, pages={26296--26306}, year={2024}}
@article{bai2025qwen25vl,
  author={Bai, Shuai and Chen, Keqin and Liu, Xuejing and Wang, Jialin and Ge, Wenbin and Song, Sibo and others},
  title={{Qwen2.5-VL} technical report}, journal={arXiv preprint arXiv:2502.13923}, year={2025}}
@article{marafioti2025smolvlm,
  author={Marafioti, Andr{\'e}s and Zohar, Orr and Farr{\'e}, Miquel and Noyan, Merve and Bakouch, Elie and Cuenca, Pedro and others},
  title={{SmolVLM}: Redefining small and efficient multimodal models}, journal={arXiv preprint arXiv:2504.05299}, year={2025}}
@inproceedings{suvorov2022lama,
  author={Suvorov, Roman and Logacheva, Elizaveta and Mashikhin, Anton and Remizova, Anastasia and Ashukha, Arsenii and Silvestrov, Aleksei and others},
  title={Resolution-robust large mask inpainting with {Fourier} convolutions}, booktitle={Proc. IEEE/CVF WACV}, pages={2149--2159}, year={2022}}
@article{telea2004,
  author={Telea, Alexandru}, title={An image inpainting technique based on the fast marching method}, journal={Journal of Graphics Tools}, volume={9}, number={1}, pages={23--34}, year={2004}}
@inproceedings{rombach2022,
  author={Rombach, Robin and Blattmann, Andreas and Lorenz, Dominik and Esser, Patrick and Ommer, Bj{\"o}rn},
  title={High-resolution image synthesis with latent diffusion models}, booktitle={Proc. IEEE/CVF CVPR}, pages={10684--10695}, year={2022}}
@inproceedings{minderer2022owlvit,
  author={Minderer, Matthias and Gritsenko, Alexey and Stone, Austin and Neumann, Maxim and Weissenborn, Dirk and Dosovitskiy, Alexey and others},
  title={Simple open-vocabulary object detection}, booktitle={Proc. ECCV}, pages={728--755}, year={2022}}
@inproceedings{dettmers2023qlora,
  author={Dettmers, Tim and Pagnoni, Artidoro and Holtzman, Ari and Zettlemoyer, Luke},
  title={{QLoRA}: Efficient finetuning of quantized {LLMs}}, booktitle={Advances in Neural Information Processing Systems}, volume={36}, year={2023}}
@inproceedings{hu2022lora,
  author={Hu, Edward J. and Shen, Yelong and Wallis, Phillip and Allen-Zhu, Zeyuan and Li, Yuanzhi and Wang, Shean and Wang, Lu and Chen, Weizhu},
  title={{LoRA}: Low-rank adaptation of large language models}, booktitle={Proc. ICLR}, year={2022}}
@inproceedings{rafailov2023dpo,
  author={Rafailov, Rafael and Sharma, Archit and Mitchell, Eric and Ermon, Stefano and Manning, Christopher D. and Finn, Chelsea},
  title={Direct preference optimization: Your language model is secretly a reward model}, booktitle={Advances in Neural Information Processing Systems}, volume={36}, year={2023}}
@article{saerens2002,
  author={Saerens, Marco and Latinne, Patrice and Decaestecker, Christine},
  title={Adjusting the outputs of a classifier to new a priori probabilities: a simple procedure}, journal={Neural Computation}, volume={14}, number={1}, pages={21--41}, year={2002}}
@inproceedings{guo2017,
  author={Guo, Chuan and Pleiss, Geoff and Sun, Yu and Weinberger, Kilian Q.},
  title={On calibration of modern neural networks}, booktitle={Proc. ICML}, pages={1321--1330}, year={2017}}
@article{dersimonian1986,
  author={DerSimonian, Rebecca and Laird, Nan}, title={Meta-analysis in clinical trials}, journal={Controlled Clinical Trials}, volume={7}, number={3}, pages={177--188}, year={1986}}
@article{schuirmann1987,
  author={Schuirmann, Donald J.}, title={A comparison of the two one-sided tests procedure and the power approach for assessing the equivalence of average bioavailability}, journal={Journal of Pharmacokinetics and Biopharmaceutics}, volume={15}, number={6}, pages={657--680}, year={1987}}
@inproceedings{lin2014coco,
  author={Lin, Tsung-Yi and Maire, Michael and Belongie, Serge and Hays, James and Perona, Pietro and Ramanan, Deva and Doll{\'a}r, Piotr and Zitnick, C. Lawrence},
  title={Microsoft {COCO}: Common objects in context}, booktitle={Proc. ECCV}, pages={740--755}, year={2014}}
@inproceedings{gardner2020contrast,
  author={Gardner, Matt and Artzi, Yoav and Basmova, Victoria and Berant, Jonathan and Bogin, Ben and Chen, Sihao and others},
  title={Evaluating models' local decision boundaries via contrast sets}, booktitle={Findings of EMNLP}, pages={1307--1323}, year={2020}}
@inproceedings{kaushik2020,
  author={Kaushik, Divyansh and Hovy, Eduard and Lipton, Zachary C.},
  title={Learning the difference that makes a difference with counterfactually-augmented data}, booktitle={Proc. ICLR}, year={2020}}
@inproceedings{he2025rohe,
  author={He, Yixiao and Sun, Haifeng and Ren, Pengfei and Wang, Jingyu and Wang, Huazheng and Qi, Qi and Zhuang, Zirui and Wang, Jing},
  title={Evaluating and mitigating object hallucination in large vision-language models: Can they still see removed objects?},
  booktitle={Proc. NAACL (Long Papers)}, pages={6841--6858}, year={2025}, doi={10.18653/v1/2025.naacl-long.349}}
@article{yang2026hiidpo,
  author={Yang, Yilin and Guo, Zhenghui and Wang, Yuke and Gnawali, Omprakash and Di, Sheng and Zhang, Chengming},
  title={{HII-DPO}: Eliminate hallucination via accurate hallucination-inducing counterfactual images},
  journal={arXiv preprint arXiv:2602.10425}, year={2026}}
@article{li2025hallusegbench,
  author={Li, Xinzhuo and Juvekar, Adheesh and Zhang, Jiaxun and Liu, Xingyou and Wahed, Muntasir and Nguyen, Kiet A. and Shen, Yifan and Yu, Tianjiao and Lourentzou, Ismini},
  title={Counterfactual segmentation reasoning: Diagnosing and mitigating pixel-grounding hallucination ({HalluSegBench})},
  journal={arXiv preprint arXiv:2506.21546}, year={2025}}
@article{li2025popev2,
  author={Li, Yifan and Zhou, Kun and Zhao, Wayne Xin and Fang, Lei and Wen, Ji-Rong},
  title={Analyzing and mitigating object hallucination: A training bias perspective ({POPEv2}, {Obliviate})},
  journal={arXiv preprint arXiv:2508.04567}, year={2025}}


In [ ]:
res = C.check_all(open('/kaggle/working/refs.bib').read())
lines = []
for r in res:
    lines.append(f"{r['flag']:<38}{r['key']:<22}sim {r['similarity']:.2f} | bib: {r['bib_title'][:60]} ({r['bib_year']}) {r['bib_venue'][:30]}\n{'':<60}found: {r['found_title'][:60]} ({r['found_year']}) {r['found_venue'][:34]} [{r['src'][:40]}]")
txt = '\n'.join(lines); open('/kaggle/working/reference_check.txt', 'w').write(txt); print(txt)
print('\nOK:', sum(r['flag']=='OK' for r in res), 'of', len(res))

## 5. Licences of the models used (from their Hugging Face model cards)

In [ ]:
rows = C.licences()
txt = '\n'.join(f'{m:<56}{lic:<14}{name:<24}{link[:50]:<50}{g}' for m, lic, name, link, g in rows)
open('/kaggle/working/model_licences.txt', 'w').write(txt); print(txt)

In [ ]:
!cd /kaggle/working && zip -q checks_out.zip literature_grep.txt reference_check.txt model_licences.txt && ls -la checks_out.zip